# Adaptive Optimizers — RMSProp, Adam, AdamW, Lion

**Author:** Siddharth Bokka  
**Dataset:** MNIST (MLP training)  
**Frameworks:** Pure NumPy (update-rule demos) + PyTorch (real training) + MLflow (tracking)  

---

## What This Notebook Covers

| Optimizer | Key Innovation | Paper |
|---|---|---|
| RMSProp | Divide LR by RMS of recent gradients | Hinton, 2012 (lecture notes) |
| Adam | Momentum + RMSProp + bias correction | Kingma & Ba, 2014 |
| AdamW | Adam + decoupled weight decay | Loshchilov & Hutter, 2019 |
| Lion | Sign of gradient + momentum | Chen et al., Google Brain, 2023 |

**Part A** — NumPy from-scratch: visualize each optimizer's trajectory on the  
Rosenbrock surface to understand *why* adaptive methods navigate it differently.  

**Part B** — PyTorch training: run all four on the same MLP/MNIST task,  
same random seed, compare convergence speed and final accuracy.  
All runs logged to MLflow for structured comparison.

---

**Critical distinction explained in detail:**  
Why AdamW ≠ Adam+L2 — and why it matters for Transformer training.

## Step 1 — Setup

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch

repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

logging.basicConfig(
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
    level=logging.INFO,
)

matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 12})
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
from gdo.optimizers import RMSProp, Adam, AdamW, Lion
from gdo.landscapes import Rosenbrock, LandscapePlotter
from gdo.config import ExperimentConfig
from gdo.training import Trainer
from gdo.experiment import ExperimentLogger

print('gdo imports successful.')

---
## Part A — NumPy Trajectories on the Rosenbrock Surface

### Step 2 — Why Adaptive Methods Navigate Differently

Each adaptive optimizer maintains an internal estimate of the gradient scale  
per parameter.  When one dimension has consistently large gradients, the  
effective LR in that dimension is reduced.  This is exactly what helps on  
the Rosenbrock banana where $\partial f / \partial y$ is much larger than  
$\partial f / \partial x$ near the valley floor.

In [ ]:
rosen = Rosenbrock()
START = (-1.5, 1.0)
N_STEPS = 2000

numpy_opts = [
    RMSProp(lr=0.003, alpha=0.99),
    Adam(lr=0.01, beta1=0.9, beta2=0.999),
    AdamW(lr=0.01, beta1=0.9, beta2=0.999, weight_decay=0.01),
    Lion(lr=0.001, beta1=0.9, beta2=0.99),
]

for opt in numpy_opts:
    rosen.run_optimizer(opt, start=START, n_steps=N_STEPS)
    end = opt.trajectory[-1]
    print(
        f'{opt.name:<35} '
        f'final=({end[0]:.3f}, {end[1]:.3f})  '
        f'loss={opt.loss_history[-1]:.4f}  '
        f'steps={opt.n_steps}'
    )

In [ ]:
fig = LandscapePlotter.trajectory_comparison(
    surface=rosen,
    optimizers=numpy_opts,
    figsize=(14, 5),
)
plt.show()

fig = LandscapePlotter.convergence_curves(
    optimizers=numpy_opts,
    title='Rosenbrock — Adaptive Optimizers Convergence',
    log_scale=True,
)
plt.show()

---
### Step 3 — Adam vs AdamW: The Weight Decay Distinction

This is the most commonly misunderstood difference in modern deep learning.

#### Adam with L2 regularization (incorrect weight decay)
Adding L2 reg to the loss means $g = \nabla L + \lambda \theta$  
Then Adam computes moments from this combined gradient:  
$$m \leftarrow \beta_1 m + (1-\beta_1)(\nabla L + \lambda \theta)$$  
The weight decay term gets **divided by the adaptive scale** $\hat{v}$,  
which means parameters with large gradients get **less** weight decay.  
This is not the intended behavior.

#### AdamW (correct decoupled weight decay)
Weight decay is applied **directly to the parameters**, before the gradient step:  
$$\theta \leftarrow (1 - \alpha \lambda) \cdot \theta - \alpha \cdot \hat{m} / (\sqrt{\hat{v}} + \epsilon)$$  
The gradient moments are computed from $\nabla L$ only (no $\lambda \theta$).  
This is what BERT, GPT, ViT, and all modern Transformer training use.

In [ ]:
# Demonstrate the numerical difference between Adam+L2 and AdamW
# on a single parameter update step

lr = 0.001
wd = 0.1  # large weight decay to make the difference visible
param = np.array([2.0])  # initial parameter
grad = np.array([0.5])   # gradient at this point

# Adam with L2 (wrong)
grad_with_l2 = grad + wd * param  # l2 reg adds lambda * theta to gradient
adam_l2 = Adam(lr=lr)
new_param_adam_l2 = adam_l2.step(param.copy(), grad_with_l2)

# AdamW (correct)
adamw = AdamW(lr=lr, weight_decay=wd)
new_param_adamw = adamw.step(param.copy(), grad)

print(f'Initial parameter:       {param[0]:.6f}')
print(f'Gradient:                {grad[0]:.6f}')
print(f'Weight decay (λ):        {wd}')
print()
print(f'Adam+L2 → new param:     {new_param_adam_l2[0]:.6f}')
print(f'AdamW   → new param:     {new_param_adamw[0]:.6f}')
print(f'Difference:              {abs(new_param_adamw[0] - new_param_adam_l2[0]):.6f}')
print()
print('In Adam+L2, weight decay is scaled by 1/sqrt(v̂) — not what you want.')
print('In AdamW, weight decay is applied at full strength independently of gradients.')

---
## Part B — PyTorch Training on MNIST (Config-Driven)

### Step 4 — Run All Four Optimizers, Same Task, Same Seed

Each experiment is loaded from its YAML config file.  
All runs log to the same MLflow experiment: `optimizer-comparison-mnist`.  
After training, open `mlflow ui` in the repo root to compare runs in a table.

In [ ]:
configs_dir = repo_root / 'configs'

config_files = [
    configs_dir / 'rmsprop_mnist.yaml',
    configs_dir / 'adam_mnist.yaml',
    configs_dir / 'adamw_mnist.yaml',
    configs_dir / 'lion_mnist.yaml',
]

results = {}

for cfg_path in config_files:
    cfg = ExperimentConfig.from_yaml(cfg_path)
    print(f'\n{'='*60}')
    print(f'Running: {cfg.mlflow.run_name}')
    print(f'{'='*60}')

    trainer = Trainer.from_config(cfg)

    with ExperimentLogger(cfg.mlflow) as log:
        result = trainer.fit()
        log.log_run(result, cfg)

    results[cfg.optimizer.name.value] = result
    print(f'Done — best_val_acc={result.best_val_acc:.4f} at epoch {result.best_epoch}')

### Step 5 — Compare Results

In [ ]:
print(f'{'Optimizer':<15} {'Best Val Acc':<15} {'Best Val Loss':<15} {'Best Epoch':<12} {'Train Time (s)'}')
print('-' * 72)
for name, result in results.items():
    print(
        f'{name:<15} '
        f'{result.best_val_acc:<15.4f} '
        f'{result.best_val_loss:<15.4f} '
        f'{result.best_epoch:<12} '
        f'{result.total_train_time_s:.1f}'
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
for i, (name, result) in enumerate(results.items()):
    color = colors[i % len(colors)]
    epochs = list(range(len(result.val_losses)))
    axes[0].plot(epochs, result.train_losses, label=name, color=color, linewidth=2)
    axes[1].plot(epochs, result.val_accs, label=name, color=color, linewidth=2)

for ax, title, ylabel in zip(
    axes,
    ['Train Loss', 'Val Accuracy'],
    ['Loss', 'Accuracy']
):
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle('Adaptive Optimizers — MNIST MLP (same seed, same architecture)', fontsize=14)
fig.tight_layout()
plt.show()

---
## Key Takeaways

1. **RMSProp** adapts the learning rate per parameter using a running average  
   of squared gradients. Historically important — the stepping stone to Adam.

2. **Adam** combines RMSProp (second moment) with Momentum (first moment) and  
   bias correction. It is the industry default optimizer as of 2024.  
   Works well with the default LR of 0.001 across most tasks.

3. **AdamW** fixes Adam's weight decay bug. For Transformers, AdamW consistently  
   outperforms Adam because proper decoupled weight decay improves generalization.  
   Use AdamW as the default for any Transformer training.

4. **Lion** stores only one moment vector (vs two for Adam), making it more  
   memory-efficient at scale. It uses the sign of the update direction rather  
   than the magnitude, which acts as a form of implicit gradient clipping.  
   Best with a ~10x smaller LR than Adam and strong weight decay.

5. Open `mlflow ui` in the repo root to see all runs in a structured comparison table.

**Next:** Notebook 3 — Learning Rate Scheduling and Advanced Techniques